In [12]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Carregar a base de dados
df = pd.read_parquet("data.parquet")

# Indicadores solicitados
indicadores = [
    "ceb_afd_4_1",
    "ceb_afd_4_2",
    "ceb_escolas_4_2",
    "ceb_escolas_4_3",
    "ceb_escolas_4_5"
]
df_filtrado = df[df["id_indicator"].isin(indicadores)]

#cada municipio = cada linha, cada indicador = cada coluna
df_wide = df_filtrado.pivot(
    index="id_local",
    columns="id_indicator",
    values="value"
)

In [13]:
print(df_wide.isna().sum())

id_indicator
ceb_afd_4_1        41
ceb_afd_4_2        26
ceb_escolas_4_2     0
ceb_escolas_4_3     0
ceb_escolas_4_5     0
dtype: int64


In [ ]:


df_wide = df_wide.dropna()

scaler = StandardScaler()
X = scaler.fit_transform(df_wide)

In [11]:
modelo = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

modelo.fit(X)

# Predições
df_wide["anomalia"] = modelo.predict(X)
df_wide["score"] = modelo.decision_function(X)

# Municípios anômalos
anomalias = (
    df_wide[df_wide["anomalia"] == -1]
    .sort_values("score")
)

print(f"Quantidade de municípios anômalos: {len(anomalias)}")
print(anomalias.head(10))

Quantidade de municípios anômalos: 9
id_indicator  ceb_afd_4_1  ceb_afd_4_2  ceb_escolas_4_2  ceb_escolas_4_3  \
id_local                                                                   
2506103            0.4000       0.2222           0.0769           0.7692   
2514602            0.0000       1.0000           0.0000           0.0000   
2513604            0.0177       0.0667           0.2500           0.2500   
2500502            0.0000       0.5320           0.3333           0.6667   
2511905            0.0000       0.9900           0.6667           0.5000   
2512788            1.0000       1.0000           0.0000           0.5000   
2500536            0.3075       0.3333           0.0000           0.5000   
2508307            0.3330       0.2000           1.0000           0.0000   
2504504            0.0000       1.0000           0.5000           0.0000   

id_indicator  ceb_escolas_4_5  anomalia     score  
id_local                                           
2506103               